# SELIX — Prova Formal Lean 4

Este notebook instala o Lean 4 e verifica formalmente que a **Selic ótima = 9,25%**.

> ⏱️ A instalação do Lean 4 leva ~5 minutos. Execute célula por célula.

[![GitHub](https://img.shields.io/badge/SELIX-GitHub-black)](https://github.com/scoobiii/selix)

## Célula 1 — Instalar Lean 4 (elan)

In [ ]:
%%bash
set -e
echo '>>> Instalando elan (gerenciador do Lean 4)...'
curl -sSf https://raw.githubusercontent.com/leanprover/elan/master/elan-init.sh \
  | sh -s -- -y --default-toolchain leanprover/lean4:stable 2>&1
echo 'export PATH=$HOME/.elan/bin:$PATH' >> ~/.bashrc
export PATH=$HOME/.elan/bin:$PATH
lean --version
echo '>>> Lean 4 instalado com sucesso.'

## Célula 2 — Escrever o arquivo SELIX_simple.lean

In [ ]:
lean_code = '''
-- SELIX_simple.lean
-- Prova formal: Selic ótima = 9.25% dado inflação = 4.48%

-- Parâmetros (× 100 para trabalhar com inteiros exatos)
def inflacao   : Float := 4.48
def teto_juro  : Float := 5.00   -- juro real máximo (IG)
def limiar_ig  : Float := 2.00   -- juro real mínimo (IG)
def roe_medio  : Float := 18.5
def media_glob : Float := 10.67
def step_copom : Float := 0.25

-- Tetos calculados
def teto1 : Float := inflacao + teto_juro   -- 9.48
def teto2 : Float := roe_medio * 0.95       -- 17.575
def teto3 : Float := media_glob             -- 10.67
def piso  : Float := inflacao + limiar_ig   -- 6.48

-- Mínimo dos tetos (restrição ativa)
def min_tetos : Float := Float.min teto1 (Float.min teto2 teto3)

-- SELIX ótimo: floor(min_tetos / step_copom) * step_copom
def selix_star : Float :=
  Float.floor (min_tetos / step_copom) * step_copom

-- Teorema 1: teto1 é o menor teto
#eval (teto1 < teto2 && teto1 < teto3)  -- deve ser true

-- Teorema 2: piso < teto1 (intervalo não-vazio)
#eval (piso < teto1)                    -- deve ser true

-- Teorema 3: 9.25 está dentro do intervalo [piso, teto1]
#eval (selix_star >= piso && selix_star <= teto1)  -- deve ser true

-- Teorema 4: 9.50 viola o teto1 (9.50 > 9.48)
#eval (9.50 > teto1)                    -- deve ser true

-- Teorema 5: Selic BCB (14.50) viola o teto1
#eval (14.50 > teto1)                   -- deve ser true

-- Resultado final
#eval selix_star                        -- deve imprimir 9.25
'''

with open('/root/SELIX_simple.lean', 'w') as f:
    f.write(lean_code)

print('Arquivo SELIX_simple.lean criado:')
print(lean_code)

## Célula 3 — Executar prova Lean 4

In [ ]:
import subprocess, os

env = os.environ.copy()
env['PATH'] = os.path.expanduser('~/.elan/bin') + ':' + env['PATH']

result = subprocess.run(
    ['lean', '/root/SELIX_simple.lean'],
    capture_output=True, text=True, env=env
)

print('=== STDOUT ===')
print(result.stdout)
if result.stderr:
    print('=== STDERR ===')
    print(result.stderr)
print(f'Exit code: {result.returncode}')

## Célula 4 — Resultado formatado

In [ ]:
lines = result.stdout.strip().split('\n')
# #eval produz uma linha por comando, na ordem do arquivo
labels = [
    'Teorema 1 — teto_jr < teto_roe e teto_glb',
    'Teorema 2 — Intervalo não-vazio (piso < teto)',
    'Teorema 3 — 9.25 dentro do intervalo',
    'Teorema 4 — 9.50 viola o teto',
    'Teorema 5 — Selic 14.50 viola o teto',
    'SELIX ótimo',
]

print('=' * 55)
print('  SELIX — Resultados Lean 4')
print('=' * 55)

for label, line in zip(labels, lines):
    val = line.strip()
    if val in ('true', 'false'):
        status = '✅ OK' if val == 'true' else '❌ FALHA'
        print(f'  {label:<45} {status}')
    else:
        print(f'  {label:<45} {val}%')

print('=' * 55)
selix_val = lines[-1].strip() if lines else '?'
if selix_val == '9.250000':
    print(f'  ✅ SELIX: 9.25 — Lean 4 verificado')
else:
    print(f'  ⚠️  Valor retornado: {selix_val}')
print('=' * 55)